# Extracción de biomarcadores estructurales con FreeSurfer y fMRIPrep

El proceso se organiza en cuatro etapas independientes:

1. **Extracción de volúmenes de tejido por ROI** — mapeo de mapas de probabilidad de GM, WM y CSF sobre el atlas Harvard-Oxford
2. **Extracción de grosores corticales** — lectura de `aparc.stats` para ambos hemisferios
3. **Extracción de volúmenes subcorticales** — lectura de `aseg.stats` con normalización por ICV
4. **Dataset combinado** — integración de volúmenes subcorticales, grosores y área superficial en un único CSV por grupo y sexo

## 1. Extracción de volúmenes de tejido por ROI — Atlas Harvard-Oxford

Las imágenes preprocesadas por fMRIPrep incluyen mapas de probabilidad de tejido en espacio MNI152 para tres clases:

| Tejido | Código | Descripción |
|---|---|---|
| Sustancia gris | `GM` | Principal tejido neuronal cortical y subcortical |
| Sustancia blanca | `WM` | Tractos de fibras mielinizadas |
| Líquido cefalorraquídeo | `CSF` | Fluido de espacios ventriculares y subaracnoideos |

Para cada sujeto y tejido se realizan tres operaciones:

1. **Remuestreo del atlas** al espacio de la imagen de tejido mediante interpolación por vecino más próximo
2. **Extracción del volumen regional** como suma de probabilidades dentro de cada ROI: $V_{ROI} = \sum_{v \in ROI} p(v)$
3. **Normalización por ICV** — el ICV se estima como la suma de probabilidades de GM + WM + CSF sobre todo el volumen

Se emplean dos atlas complementarios de Harvard-Oxford:

- **Cortical** (`HarvardOxford-cort-maxprob-thr25-1mm`)
- **Subcortical** (`HarvardOxford-sub-maxprob-thr25-1mm`)

## 2. Extracción de grosores corticales — `aparc.stats`

FreeSurfer genera por cada sujeto dos archivos `aparc.stats` (uno por hemisferio) con las estadísticas de la parcelación cortical según el atlas Desikan-Killiany. De cada archivo se extrae la columna de **grosor cortical promedio** (`ThickAvg`, columna 5).

El nombre de cada variable resultante sigue el formato: {región}_{hemisferio}_thickness donde `hemisferio` es `L` (izquierdo) o `R` (derecho). Las variables de grosor **no se normalizan por ICV**, ya que el grosor cortical es una medida de longitud independiente del volumen total.

## 3. Extracción de volúmenes subcorticales — `aseg.stats`

El archivo `aseg.stats` de FreeSurfer contiene los volúmenes de las estructuras subcorticales segmentadas automáticamente. Se extraen dos tipos de información:

- **ICV** (*Estimated Total Intracranial Volume*) — leído de la línea de cabecera que contiene `EstimatedTotalIntraCranialVol`
- **Volúmenes subcorticales** — nombre de la estructura en la columna 5, volumen en la columna 4

Todos los volúmenes extraídos se **normalizan por el ICV del propio sujeto** para corregir las diferencias individuales en tamaño craneal:

$$V_{norm} = \frac{V_{estructura}}{ICV}$$

## 4. Dataset combinado — Volúmenes subcorticales + grosores + área superficial

Este script genera el dataset final que combina en un único CSV por grupo y género tres tipos de biomarcadores extraídos de FreeSurfer:

| Fuente | Medida | Normalización |
|---|---|---|
| `aseg.stats` | Volúmenes subcorticales | Sí, por ICV |
| `aparc.stats` | Grosor cortical (`ThickAvg`) | No |
| `aparc.stats` | Área superficial (`SurfArea`) | No |

<br>

La normalización se aplica selectivamente: solo las columnas de volumen (procedentes de `aseg.stats`) se dividen por ICV. Los grosores y áreas superficiales se guardan en sus unidades originales (mm y mm² respectivamente), ya que son medidas morfológicas independientes del tamaño craneal.

El nombre de cada archivo de salida sigue el patrón: volumenes_grosores_corticales_y_surfarea_{grupo}_{genero}.csv

Estos CSVs son los **datasets de entrada finales** para el pipeline de clasificación.

# 1) csf_gm_wm_harvard_oxford

In [2]:
import nibabel as nib
import numpy as np
import pandas as pd
from nilearn.image import resample_to_img
import os
import glob
import warnings

warnings.filterwarnings('ignore')

# Ruta base que contiene las carpetas de cada grupo
ruta_base_grupos = r"\\wsl.localhost\Ubuntu\home\joselito"  # Ajusta según tu estructura

# Definir los grupos a procesar (Alzheimer, EMCI, LMCI, Controles Normales)
grupos = ["AD", "EMCI", "LMCI", "CN"]

# Definir los géneros a procesar (masculino y femenino)
generos = ["m", "f"]

# Diccionario para mapear cada grupo y género a su carpeta específica
rutas_grupos_generos = {
    ("AD", "m"): os.path.join(ruta_base_grupos, "FMRIPREP_AD_m_MRI"),
    ("AD", "f"): os.path.join(ruta_base_grupos, "FMRIPREP_AD_f_MRI"),
    ("EMCI", "m"): os.path.join(ruta_base_grupos, "FMRIPREP_EMCI_m_MRI"),
    ("EMCI", "f"): os.path.join(ruta_base_grupos, "FMRIPREP_EMCI_f_MRI"),
    ("LMCI", "m"): os.path.join(ruta_base_grupos, "FMRIPREP_LMCI_m_MRI"),
    ("LMCI", "f"): os.path.join(ruta_base_grupos, "FMRIPREP_LMCI_f_MRI"),
    ("CN", "m"): os.path.join(ruta_base_grupos, "FMRIPREP_CN_m_MRI"),
    ("CN", "f"): os.path.join(ruta_base_grupos, "FMRIPREP_CN_f_MRI")
}

# Rutas de los atlas cortical y subcortical de Harvard-Oxford
atlas_cort = r"D:\progreso actual\Atlas\HarvardOxford-cort-maxprob-thr25-1mm.nii.gz"
labels_cort_path = r"D:\progreso actual\Atlas\HarvardOxford-cort-maxprob-thr25-1mm.txt"

atlas_sub = r"D:\progreso actual\Atlas\HarvardOxford-sub-maxprob-thr25-1mm.nii.gz"
labels_sub_path = r"D:\progreso actual\Atlas\HarvardOxford-sub-maxprob-thr25-1mm.txt"

# Función para leer los nombres de las regiones del atlas desde un archivo de texto
def leer_nombres_atlas(ruta_txt):
    """
    Lee un archivo de texto con los nombres de las regiones del atlas.
    
    Parámetros:
    -----------
    ruta_txt : str
        Ruta al archivo .txt que contiene los nombres de las regiones
    
    Retorna:
    --------
    list
        Lista con los nombres de las regiones (una por línea)
    """
    with open(ruta_txt, "r", encoding="utf-8") as f:
        return [linea.strip() for linea in f if linea.strip()]

# Función para extraer el volumen de cada región del atlas para un tejido específico
def extraer_volumen_por_region(imagen_probseg, imagen_atlas, lista_etiquetas, prefijo_tejido):
    """
    Extrae el volumen (suma de probabilidades) de cada región del atlas para un tejido específico.
    
    Parámetros:
    -----------
    imagen_probseg : nibabel image
        Imagen de probabilidad del tejido (GM, WM o CSF)
    imagen_atlas : nibabel image
        Imagen del atlas con regiones etiquetadas
    lista_etiquetas : list
        Lista con los nombres de las regiones del atlas
    prefijo_tejido : str
        Prefijo para nombrar las columnas (ej: "GM_Cort", "WM_Sub")
    
    Retorna:
    --------
    dict
        Diccionario con los volúmenes por región (key: nombre_región, value: volumen)
    """
    # Alinear el atlas a la imagen del tejido (mismo espacio y resolución)
    atlas_alineado = resample_to_img(imagen_atlas, imagen_probseg, interpolation="nearest")
    # Obtener los datos numéricos de las imágenes
    data_tejido = imagen_probseg.get_fdata()
    data_atlas = atlas_alineado.get_fdata()
    
    resultados = {}
    # Iterar sobre cada región del atlas (las etiquetas empiezan en 1)
    for i, nombre in enumerate(lista_etiquetas, start=1):
        # Crear máscara binaria de la región actual
        mascara_roi = data_atlas == i
        if np.any(mascara_roi):
            # Sumar los valores de probabilidad del tejido dentro de la región
            suma_prob = np.sum(data_tejido[mascara_roi])
        else:
            suma_prob = 0
        # Guardar resultado con el formato: {prefijo}_{nombre_región}
        resultados[f"{prefijo_tejido}_{nombre}"] = suma_prob
    return resultados

# Función para obtener la lista de sujetos válidos en una ruta base (aquellos que tienen una carpeta anat)
def obtener_sujetos_validos(ruta_base):
    """
    Obtiene la lista de sujetos válidos en una ruta base.
    Un sujeto es válido si tiene una carpeta 'anat' dentro de su directorio.
    
    Parámetros:
    -----------
    ruta_base : str
        Ruta base que contiene las carpetas de los sujetos (formato sub-*)
    
    Retorna:
    --------
    list
        Lista de rutas completas de los sujetos válidos
    """
    # Buscar todas las carpetas que comienzan con "sub-"
    todos_items = glob.glob(os.path.join(ruta_base, "sub-*"))
    # Filtrar aquellas que contengan una subcarpeta "anat"
    sujetos = [s for s in todos_items if os.path.isdir(os.path.join(s, "anat"))]
    print(f"Sujetos válidos encontrados: {len(sujetos)}")
    return sujetos

# Cargar atlas y etiquetas (una sola vez para todos los grupos y géneros)
print("Cargando atlas y etiquetas...")
atlas_cort_img = nib.load(atlas_cort)
atlas_sub_img = nib.load(atlas_sub)
labels_cort = leer_nombres_atlas(labels_cort_path)
labels_sub = leer_nombres_atlas(labels_sub_path)
print(f"Atlas cortical: {len(labels_cort)} regiones")
print(f"Atlas subcortical: {len(labels_sub)} regiones")

# Tipos de tejido a procesar (Sustancia Gris, Sustancia Blanca, Líquido Cefalorraquídeo)
tipos_tejido = ["GM", "WM", "CSF"]

# Procesar cada combinación de grupo y género por separado
for grupo in grupos:
    for genero in generos:
        print(f"\n{'='*60}")
        print(f"PROCESANDO GRUPO: {grupo} - GÉNERO: {genero.upper()}")
        print(f"{'='*60}")
        
        # Obtener la ruta específica para este grupo y género
        ruta_mri_preproc = rutas_grupos_generos.get((grupo, genero))
        
        # Verificar si la ruta del grupo existe
        if not os.path.exists(ruta_mri_preproc):
            print(f"ADVERTENCIA: Ruta no encontrada para {grupo} ({genero}): {ruta_mri_preproc}")
            continue
        
        # Obtener lista de sujetos válidos en esta ruta
        sujetos = obtener_sujetos_validos(ruta_mri_preproc)
        resultados_finales = []  # Lista para almacenar los datos de todos los sujetos
        
        # Procesar cada sujeto individualmente
        for idx_sujeto, sujeto in enumerate(sujetos, start=1):
            # Extraer el ID del sujeto (nombre de la carpeta)
            id_sujeto = os.path.basename(sujeto)
            print(f"Procesando: Sujeto {idx_sujeto}...", end=" ", flush=True)
            
            # Inicializar diccionario para este sujeto (solo con el ID)
            fila_sujeto = {"subject": id_sujeto}  # Sin columnas group ni gender

            # Cálculo del ICV (Intracraneal Volume) sumando las probabilidades de GM, WM y CSF
            icv = 0.0
            for tejido in tipos_tejido:
                # Construir nombre del archivo de probabilidad del tejido
                archivo_tejido = f"{id_sujeto}_space-MNI152NLin2009cAsym_label-{tejido}_probseg.nii.gz"
                ruta_archivo = os.path.join(sujeto, "anat", archivo_tejido)

                if os.path.exists(ruta_archivo):
                    # Cargar imagen y sumar todas las probabilidades
                    img_tejido = nib.load(ruta_archivo)
                    icv += np.sum(img_tejido.get_fdata())

            # Guardar el ICV calculado
            fila_sujeto["ICV"] = icv

            # Extracción de ROIs (Regiones de Interés) de cada tejido
            for tejido in tipos_tejido:
                # Construir nombre del archivo de probabilidad del tejido
                archivo_tejido = f"{id_sujeto}_space-MNI152NLin2009cAsym_label-{tejido}_probseg.nii.gz"
                ruta_archivo = os.path.join(sujeto, "anat", archivo_tejido)

                if os.path.exists(ruta_archivo):
                    # Cargar la imagen de probabilidad del tejido
                    img_tejido = nib.load(ruta_archivo)

                    # Extraer volúmenes de regiones corticales
                    datos_cort = extraer_volumen_por_region(
                        img_tejido, atlas_cort_img, labels_cort, f"{tejido}_Cort"
                    )

                    # Normalizar los valores corticales por ICV
                    datos_cort = {
                        k: (v / icv if icv > 0 else 0)
                        for k, v in datos_cort.items()
                    }
                    fila_sujeto.update(datos_cort)

                    # Extraer volúmenes de regiones subcorticales
                    datos_sub = extraer_volumen_por_region(
                        img_tejido, atlas_sub_img, labels_sub, f"{tejido}_Sub"
                    )

                    # Normalizar los valores subcorticales por ICV
                    datos_sub = {
                        k: (v / icv if icv > 0 else 0)
                        for k, v in datos_sub.items()
                    }
                    fila_sujeto.update(datos_sub)

                else:
                    # Advertir si falta algún archivo de tejido
                    print(f"(Falta {tejido})", end=" ")

            # Añadir los datos del sujeto a la lista de resultados
            resultados_finales.append(fila_sujeto)
            print("Hecho.")
        
        # Convertir el nombre del grupo a minúsculas para el nombre del archivo
        grupo_minuscula = grupo.lower()
        # Crear nombre del archivo CSV: csf_gm_wm_harvard_oxford_no_{grupo}_{genero}.csv
        output_csv = f"csf_gm_wm_harvard_oxford_{grupo_minuscula}_{genero}.csv"
        
        # Convertir la lista de diccionarios a DataFrame de pandas
        df_final = pd.DataFrame(resultados_finales)
        
        # Guardar el DataFrame como archivo CSV (sin incluir el índice)
        df_final.to_csv(output_csv, index=False)
        
        # Mostrar información del archivo generado
        print(f"\nGrupo {grupo} ({genero}) finalizado. Archivo creado: {output_csv}")
        print(f"Sujetos: {df_final.shape[0]}, Features: {df_final.shape[1] - 1}")  # -1 por la columna subject

# Mensaje final de finalización del proceso
print("\nProceso completamente finalizado.")

Cargando atlas y etiquetas...
Atlas cortical: 47 regiones
Atlas subcortical: 20 regiones

PROCESANDO GRUPO: AD - GÉNERO: M
Sujetos válidos encontrados: 60
Procesando: Sujeto 1... Hecho.
Procesando: Sujeto 2... Hecho.
Procesando: Sujeto 3... Hecho.
Procesando: Sujeto 4... Hecho.
Procesando: Sujeto 5... Hecho.
Procesando: Sujeto 6... Hecho.
Procesando: Sujeto 7... Hecho.
Procesando: Sujeto 8... Hecho.
Procesando: Sujeto 9... Hecho.
Procesando: Sujeto 10... Hecho.
Procesando: Sujeto 11... Hecho.
Procesando: Sujeto 12... Hecho.
Procesando: Sujeto 13... Hecho.
Procesando: Sujeto 14... Hecho.
Procesando: Sujeto 15... Hecho.
Procesando: Sujeto 16... Hecho.
Procesando: Sujeto 17... Hecho.
Procesando: Sujeto 18... Hecho.
Procesando: Sujeto 19... Hecho.
Procesando: Sujeto 20... Hecho.
Procesando: Sujeto 21... Hecho.
Procesando: Sujeto 22... Hecho.
Procesando: Sujeto 23... Hecho.
Procesando: Sujeto 24... Hecho.
Procesando: Sujeto 25... Hecho.
Procesando: Sujeto 26... Hecho.
Procesando: Sujeto 27.

# 2) grosores_corticales

In [1]:
import os
import pandas as pd
import glob

# Ruta base que contiene las carpetas de cada grupo
ruta_base_grupos = r"\\wsl.localhost\Ubuntu\home\joselito"

# Definir los grupos a procesar (Alzheimer, EMCI, LMCI, Controles Normales)
grupos = ["AD", "EMCI", "LMCI", "CN"]

# Definir los géneros a procesar (masculino y femenino)
generos = ["m", "f"]

# Función para rastrear el grosor cortical de cada región a partir del archivo aparc.stats
def parse_aparc(path, hemi):
    """
    Parsea el archivo aparc.stats de FreeSurfer para extraer grosores corticales.
    
    Parámetros:
    -----------
    path : str
        Ruta al archivo lh.aparc.stats o rh.aparc.stats
    hemi : str
        Hemisferio ("L" o "R")
    
    Retorna:
    --------
    dict
        Diccionario con los grosores corticales por región
    """
    data = {}  # Diccionario para almacenar los grosores corticales

    if not os.path.exists(path):  # Verificar si el archivo existe
        return data  # Si no existe, retornar diccionario vacío

    with open(path) as f:  # Abrir el archivo aparc.stats
        for line in f:  # Iterar línea por línea

            if line.startswith("#"):  # Saltar líneas de comentario
                continue

            parts = line.split()  # Dividir la línea por espacios

            if len(parts) < 5:  # Las líneas válidas deben tener al menos 5 elementos
                continue

            try:
                region = parts[0]  # El nombre de la región está en la 1ª columna
                thickness = float(parts[4])  # El grosor cortical está en la 5ª columna
                # Guardar el grosor con el formato: región_hemisferio_thickness
                data[f"{region}_{hemi}_thickness"] = thickness
            except:
                continue  # Si hay error, continuar con la siguiente línea

    return data  # Retornar el diccionario con todos los grosores

# Procesar cada combinación de grupo y género por separado
for grupo in grupos:  # Iterar sobre cada grupo (AD, EMCI, LMCI, CN)
    for genero in generos:  # Iterar sobre cada género (m, f)
        print(f"\n{'='*60}")
        print(f"PROCESANDO GRUPO: {grupo} - GÉNERO: {genero.upper()}")
        print(f"{'='*60}")
        
        # Construir la ruta a la carpeta de FreeSurfer para este grupo y género
        # Estructura: FMRIPREP_{GRUPO}_{GENERO}_MRI/sourcedata/freesurfer
        freesurfer_dir = f"FMRIPREP_{grupo}_{genero}_MRI"
        fs_root = os.path.join(ruta_base_grupos, freesurfer_dir, "sourcedata", "freesurfer")
        
        # Verificar si la ruta existe
        if not os.path.exists(fs_root):
            print(f"ADVERTENCIA: Ruta no encontrada para {grupo} ({genero}): {fs_root}")
            continue  # Saltar esta combinación si no existe la ruta
        
        # Obtener lista de sujetos (carpetas que empiezan con "sub-")
        subjects = glob.glob(os.path.join(fs_root, "sub-*"))
        
        if not subjects:  # Si no hay sujetos, mostrar mensaje y continuar
            print(f"No se encontraron sujetos para {grupo} ({genero})")
            continue
        
        rows = []  # Lista para almacenar los datos de todos los sujetos
        
        for idx_sujeto, sub in enumerate(subjects, start=1):  # Iterar sobre cada sujeto
            sub_id = os.path.basename(sub)  # Extraer el ID del sujeto (nombre de la carpeta)
            print(f"Procesando: Sujeto {idx_sujeto}...", end=" ", flush=True)
            stats_dir = os.path.join(sub, "stats")  # Directorio stats de FreeSurfer
            
            # Construir rutas a los archivos de estadísticas de aparc
            lh_path = os.path.join(stats_dir, "lh.aparc.stats")  # Hemisferio izquierdo
            rh_path = os.path.join(stats_dir, "rh.aparc.stats")  # Hemisferio derecho
            
            # Inicializar fila con el ID del sujeto
            row = {"subject": sub_id}
            
            # Extraer grosores del hemisferio izquierdo y añadir a la fila
            row.update(parse_aparc(lh_path, "L"))
            
            # Extraer grosores del hemisferio derecho y añadir a la fila
            row.update(parse_aparc(rh_path, "R"))
            
            rows.append(row)  # Añadir la fila a la lista de resultados
            print("Hecho.")
        
        if not rows:  # Si no se procesó ningún sujeto, continuar
            print(f"No se procesaron sujetos para {grupo} ({genero})")
            continue
        
        df = pd.DataFrame(rows)  # Convertir lista de diccionarios a DataFrame de pandas
        
        # Mostrar información del DataFrame creado
        print("\nDataFrame creado")
        print("Sujetos:", df.shape[0])  # Número de filas (sujetos)
        print("Features:", df.shape[1])  # Número de columnas (características)
        
        # Filtrar columnas que contienen "thickness" en su nombre
        thickness_cols = [c for c in df.columns if "thickness" in c.lower()]
        
        # Verificar que se encontraron columnas de grosor
        if len(thickness_cols) == 0:
            print(f"ADVERTENCIA: No se encontraron columnas de thickness para {grupo} ({genero})")
            # No usamos raise ValueError para no detener el proceso de otros grupos
            continue  # Saltar esta combinación si no hay columnas de thickness
        
        # Mostrar cuántas columnas de grosor se encontraron
        print("\nColumnas thickness:", len(thickness_cols))
        
        # Convertir grupo a minúsculas para el nombre del archivo
        grupo_minuscula = grupo.lower()
        
        # Crear nombre del archivo CSV: grosores_corticales_{grupo}_{genero}.csv
        output_csv = f"grosores_corticales_{grupo_minuscula}_{genero}.csv"
        
        # Guardar el DataFrame como archivo CSV (sin incluir el índice)
        df.to_csv(output_csv, index=False)

# Mensaje final de finalización del proceso
print("\n" + "="*60)
print("PROCESO COMPLETAMENTE FINALIZADO")
print("="*60)


PROCESANDO GRUPO: AD - GÉNERO: M
Procesando: Sujeto 1... Hecho.
Procesando: Sujeto 2... Hecho.
Procesando: Sujeto 3... Hecho.
Procesando: Sujeto 4... Hecho.
Procesando: Sujeto 5... Hecho.
Procesando: Sujeto 6... Hecho.
Procesando: Sujeto 7... Hecho.
Procesando: Sujeto 8... Hecho.
Procesando: Sujeto 9... Hecho.
Procesando: Sujeto 10... Hecho.
Procesando: Sujeto 11... Hecho.
Procesando: Sujeto 12... Hecho.
Procesando: Sujeto 13... Hecho.
Procesando: Sujeto 14... Hecho.
Procesando: Sujeto 15... Hecho.
Procesando: Sujeto 16... Hecho.
Procesando: Sujeto 17... Hecho.
Procesando: Sujeto 18... Hecho.
Procesando: Sujeto 19... Hecho.
Procesando: Sujeto 20... Hecho.
Procesando: Sujeto 21... Hecho.
Procesando: Sujeto 22... Hecho.
Procesando: Sujeto 23... Hecho.
Procesando: Sujeto 24... Hecho.
Procesando: Sujeto 25... Hecho.
Procesando: Sujeto 26... Hecho.
Procesando: Sujeto 27... Hecho.
Procesando: Sujeto 28... Hecho.
Procesando: Sujeto 29... Hecho.
Procesando: Sujeto 30... Hecho.
Procesando: Suj

# 3) volúmenes

In [2]:
import os
import pandas as pd
import glob

# Ruta base que contiene las carpetas de cada grupo
ruta_base_grupos = r"\\wsl.localhost\Ubuntu\home\joselito"

# Definir los grupos a procesar (Alzheimer, EMCI, LMCI, Controles Normales)
grupos = ["AD", "EMCI", "LMCI", "CN"]

# Definir los géneros a procesar (masculino y femenino)
generos = ["m", "f"]

# Función para rastrear el volumen de cada estructura a partir del archivo aseg.stats
def parse_aseg(path):
    """
    Parsea el archivo aseg.stats de FreeSurfer para extraer volúmenes e ICV.
    
    Parámetros:
    -----------
    path : str
        Ruta al archivo aseg.stats
    
    Retorna:
    --------
    dict
        Diccionario con los volúmenes de las estructuras y el ICV
    """
    data = {}  # Diccionario para almacenar los volúmenes extraídos

    with open(path) as f:  # Abrir el archivo aseg.stats
        for line in f:  # Iterar línea por línea
            # Buscar la línea que contiene "EstimatedTotalIntraCranialVol"
            if line.startswith("#") and "EstimatedTotalIntraCranialVol" in line:
                parts = line.replace(",", "").split()  # Eliminar comas y dividir por espacios
                try:
                    data["ICV"] = float(parts[-2])  # El ICV suele estar en la penúltima posición
                except:
                    pass  # Si falla, continuar sin guardar
                continue  # Saltar al siguiente ciclo

            if line.startswith("#"):  # Saltar líneas de comentario
                continue

            parts = line.split()  # Dividir la línea por espacios

            if len(parts) < 5:  # Las líneas válidas deben tener al menos 5 elementos
                continue

            try:
                name = parts[4]  # El nombre de la estructura está en la 5ª columna
                vol = float(parts[3])  # El volumen está en la 4ª columna
                data[name] = vol  # Guardar el volumen con el nombre de la estructura
            except:
                continue  # Si hay error, continuar con la siguiente línea

    return data  # Retornar el diccionario con todos los volúmenes

# Procesar cada combinación de grupo y género por separado
for grupo in grupos:  # Iterar sobre cada grupo (AD, EMCI, LMCI, CN)
    for genero in generos:  # Iterar sobre cada género (m, f)
        print(f"\n{'='*60}")
        print(f"PROCESANDO GRUPO: {grupo} - GÉNERO: {genero.upper()}")
        print(f"{'='*60}")
        
        # Construir la ruta a la carpeta de FreeSurfer para este grupo y género
        # Estructura: FMRIPREP_{GRUPO}_{GENERO}_MRI/sourcedata/freesurfer
        freesurfer_dir = f"FMRIPREP_{grupo}_{genero}_MRI"
        fs_root = os.path.join(ruta_base_grupos, freesurfer_dir, "sourcedata", "freesurfer")
        
        # Verificar si la ruta existe
        if not os.path.exists(fs_root):
            print(f"ADVERTENCIA: Ruta no encontrada para {grupo} ({genero}): {fs_root}")
            continue  # Saltar esta combinación si no existe la ruta
        
        # Obtener lista de sujetos (carpetas que empiezan con "sub-")
        subjects = glob.glob(os.path.join(fs_root, "sub-*"))
        
        if not subjects:  # Si no hay sujetos, mostrar mensaje y continuar
            print(f"No se encontraron sujetos para {grupo} ({genero})")
            continue
        
        rows = []  # Lista para almacenar los datos de todos los sujetos
        
        for idx_sujeto, sub in enumerate(subjects, start=1):  # Iterar sobre cada sujeto
            sub_id = os.path.basename(sub)  # Extraer el ID del sujeto (nombre de la carpeta)
            print(f"Procesando: Sujeto {idx_sujeto}...", end=" ", flush=True)
            stats_dir = os.path.join(sub, "stats")  # Directorio stats de FreeSurfer
            aseg_path = os.path.join(stats_dir, "aseg.stats")  # Ruta al archivo aseg.stats
            
            # Verificar si existe el archivo aseg.stats
            if not os.path.exists(aseg_path):
                print(f"Saltando {sub_id} — sin aseg.stats")
                continue  # Saltar este sujeto si no tiene aseg.stats
            
            # Inicializar fila con el ID del sujeto
            row = {"subject": sub_id}
            
            # Extraer volúmenes e ICV del archivo aseg.stats y añadir a la fila
            row.update(parse_aseg(aseg_path))
            
            rows.append(row)  # Añadir la fila a la lista de resultados
            print("Hecho.")
        
        if not rows:  # Si no se procesó ningún sujeto, continuar
            print(f"No se procesaron sujetos para {grupo} ({genero})")
            continue
        
        df = pd.DataFrame(rows)  # Convertir lista de diccionarios a DataFrame de pandas
        
        # Mostrar información del DataFrame creado
        print("\nDataFrame creado")
        print("Sujetos:", df.shape[0])  # Número de filas (sujetos)
        print("Features:", df.shape[1])  # Número de columnas (características)
        
        # Verificar que la columna ICV existe en el DataFrame
        if "ICV" not in df.columns:
            print(f"ADVERTENCIA: No se encontró ICV para {grupo} ({genero}) — revisar parser aseg.stats")
            continue  # Saltar esta combinación si no hay ICV
        
        # Seleccionar todas las columnas que NO son subject ni ICV
        # Estas son las columnas de volúmenes subcorticales que deben normalizarse
        cols_to_normalize = [
            col for col in df.columns 
            if col not in ["subject", "ICV"]
        ]
        
        print("\nColumnas a normalizar:", len(cols_to_normalize))
        
        # Normalizar las columnas de volumen directamente (sobrescribiendo las originales)
        for col in cols_to_normalize:
            df[col] = df[col] / df["ICV"]  # Sobrescribe la columna original con el valor normalizado
        
        print("Normalización por ICV completada (valores originales reemplazados)")
        
        # Convertir grupo a minúsculas para el nombre del archivo
        grupo_minuscula = grupo.lower()
        
        # Crear nombre del archivo CSV: volumenes_{grupo}_{genero}.csv
        output_csv = f"volumenes_{grupo_minuscula}_{genero}.csv"
        
        # Guardar el DataFrame como archivo CSV (sin incluir el índice)
        df.to_csv(output_csv, index=False)

# Mensaje final de finalización del proceso
print("\n" + "="*60)
print("PROCESO COMPLETAMENTE FINALIZADO")
print("="*60)


PROCESANDO GRUPO: AD - GÉNERO: M
Procesando: Sujeto 1... Hecho.
Procesando: Sujeto 2... Hecho.
Procesando: Sujeto 3... Hecho.
Procesando: Sujeto 4... Hecho.
Procesando: Sujeto 5... Hecho.
Procesando: Sujeto 6... Hecho.
Procesando: Sujeto 7... Hecho.
Procesando: Sujeto 8... Hecho.
Procesando: Sujeto 9... Hecho.
Procesando: Sujeto 10... Hecho.
Procesando: Sujeto 11... Hecho.
Procesando: Sujeto 12... Hecho.
Procesando: Sujeto 13... Hecho.
Procesando: Sujeto 14... Hecho.
Procesando: Sujeto 15... Hecho.
Procesando: Sujeto 16... Hecho.
Procesando: Sujeto 17... Hecho.
Procesando: Sujeto 18... Hecho.
Procesando: Sujeto 19... Hecho.
Procesando: Sujeto 20... Hecho.
Procesando: Sujeto 21... Hecho.
Procesando: Sujeto 22... Hecho.
Procesando: Sujeto 23... Hecho.
Procesando: Sujeto 24... Hecho.
Procesando: Sujeto 25... Hecho.
Procesando: Sujeto 26... Hecho.
Procesando: Sujeto 27... Hecho.
Procesando: Sujeto 28... Hecho.
Procesando: Sujeto 29... Hecho.
Procesando: Sujeto 30... Hecho.
Procesando: Suj

# 4) volúmenes_y_grosores_corticales

In [4]:
import os
import pandas as pd
import glob

# Ruta base que contiene las carpetas de cada grupo
ruta_base_grupos = r"\\wsl.localhost\Ubuntu\home\joselito"

# Definir los grupos a procesar (Alzheimer, EMCI, LMCI, Controles Normales)
grupos = ["AD", "EMCI", "LMCI", "CN"]

# Definir los géneros a procesar (masculino y femenino)
generos = ["m", "f"]


# Función para rastrear y extraer volúmenes subcorticales e ICV desde aseg.stats
def parse_aseg(path):
    """
    Parsea el archivo aseg.stats de FreeSurfer para extraer volúmenes.
    
    Parámetros:
    -----------
    path : str
        Ruta al archivo aseg.stats
    
    Retorna:
    --------
    dict
        Diccionario con los volúmenes de las estructuras
    """
    data = {}  # Diccionario para almacenar los volúmenes extraídos

    with open(path) as f:  # Abrir el archivo aseg.stats
        for line in f:  # Iterar línea por línea

            # Buscar la línea que contiene "EstimatedTotalIntraCranialVol"
            if line.startswith("#") and "EstimatedTotalIntraCranialVol" in line:
                parts = line.replace(",", "").split()  # Eliminar comas y dividir por espacios
                try:
                    data["ICV"] = float(parts[-2])  # El ICV suele estar en la penúltima posición
                except:
                    pass  # Si falla, continuar sin guardar
                continue  # Saltar al siguiente ciclo

            if line.startswith("#"):  # Saltar líneas de comentario
                continue

            parts = line.split()  # Dividir la línea por espacios

            if len(parts) < 5:  # Las líneas válidas deben tener al menos 5 elementos
                continue

            try:
                name = parts[4]  # El nombre de la estructura está en la 5ª columna
                vol = float(parts[3])  # El volumen está en la 4ª columna
                data[name] = vol  # Guardar el volumen con el nombre de la estructura
            except:
                continue  # Si hay error, continuar con la siguiente línea

    return data  # Retornar el diccionario con todos los volúmenes

# Función para rastrear y extraer grosores corticales desde aparc.stats
def parse_aparc(path, hemi):
    """
    Parsea el archivo aparc.stats de FreeSurfer para extraer grosores corticales.
    
    Parámetros:
    -----------
    path : str
        Ruta al archivo lh.aparc.stats o rh.aparc.stats
    hemi : str
        Hemisferio ("L" o "R")
    
    Retorna:
    --------
    dict
        Diccionario con los grosores corticales por región
    """
    data = {}  # Diccionario para almacenar los grosores

    if not os.path.exists(path):  # Verificar si el archivo existe
        return data  # Si no existe, retornar diccionario vacío

    with open(path) as f:  # Abrir el archivo aparc.stats
        for line in f:  # Iterar línea por línea

            if line.startswith("#"):  # Saltar líneas de comentario
                continue

            parts = line.split()  # Dividir la línea por espacios

            if len(parts) < 5:  # Las líneas válidas deben tener al menos 5 elementos
                continue

            try:
                region = parts[0]  # El nombre de la región está en la 1ª columna
                thickness = float(parts[4])  # El grosor está en la 5ª columna
                # Guardar el grosor con el formato: región_hemisferio_thickness
                data[f"{region}_{hemi}_thickness"] = thickness
            except:
                continue  # Si hay error, continuar con la siguiente línea

    return data  # Retornar el diccionario con todos los grosores

# Procesar cada combinación de grupo y género por separado
for grupo in grupos:  # Iterar sobre cada grupo (AD, EMCI, LMCI, CN)
    for genero in generos:  # Iterar sobre cada género (m, f)
        print(f"\n{'='*60}")
        print(f"PROCESANDO GRUPO: {grupo} - GÉNERO: {genero.upper()}")
        print(f"{'='*60}")
        
        # Construir la ruta a la carpeta de FreeSurfer para este grupo y género
        # Asumiendo la estructura: FMRIPREP_{GRUPO}_{GENERO}_MRI/sourcedata/freesurfer
        freesurfer_dir = f"FMRIPREP_{grupo}_{genero}_MRI"
        fs_root = os.path.join(ruta_base_grupos, freesurfer_dir, "sourcedata", "freesurfer")
        
        # Verificar si la ruta existe
        if not os.path.exists(fs_root):
            print(f"ADVERTENCIA: Ruta no encontrada para {grupo} ({genero}): {fs_root}")
            continue  # Saltar esta combinación si no existe la ruta
        
        # Obtener lista de sujetos (carpetas que empiezan con "sub-")
        subjects = glob.glob(os.path.join(fs_root, "sub-*"))
        
        if not subjects:  # Si no hay sujetos, mostrar mensaje y continuar
            print(f"No se encontraron sujetos para {grupo} ({genero})")
            continue
        
        rows = []  # Lista para almacenar los datos de todos los sujetos
        
        for idx_sujeto, sub in enumerate(subjects, start=1):  # Iterar sobre cada sujeto
            sub_id = os.path.basename(sub)  # Extraer el ID del sujeto (nombre de la carpeta)
            print(f"Procesando: Sujeto {idx_sujeto}...", end=" ", flush=True)
            stats_dir = os.path.join(sub, "stats")  # Directorio stats de FreeSurfer
            
            # Construir rutas a los archivos de estadísticas
            aseg_path = os.path.join(stats_dir, "aseg.stats")  # Volúmenes subcorticales
            lh_path = os.path.join(stats_dir, "lh.aparc.stats")  # Grosor hemisferio izquierdo
            rh_path = os.path.join(stats_dir, "rh.aparc.stats")  # Grosor hemisferio derecho
            
            # Verificar si existe el archivo aseg.stats (obligatorio)
            if not os.path.exists(aseg_path):
                print(f"Saltando {sub_id} — sin aseg.stats")
                continue  # Saltar este sujeto si no tiene aseg.stats
            
            # Inicializar fila con el ID del sujeto
            row = {"subject": sub_id}
            
            # Añadir volúmenes subcorticales e ICV desde aseg.stats
            row.update(parse_aseg(aseg_path))
            # Añadir grosores corticales del hemisferio izquierdo
            row.update(parse_aparc(lh_path, "L"))
            # Añadir grosores corticales del hemisferio derecho
            row.update(parse_aparc(rh_path, "R"))
            
            rows.append(row)  # Añadir la fila a la lista de resultados
            print("Hecho.")
        
        if not rows:  # Si no se procesó ningún sujeto, continuar
            print(f"No se procesaron sujetos para {grupo} ({genero})")
            continue
        
        df = pd.DataFrame(rows)  # Convertir lista de diccionarios a DataFrame de pandas
        
        print("\nDataFrame creado")
        print("Sujetos:", df.shape[0])  # Número de filas (sujetos)
        print("Features:", df.shape[1])  # Número de columnas (características)
        
        # Verificar que la columna ICV existe en el DataFrame
        if "ICV" not in df.columns:
            print(f"ADVERTENCIA: No se encontró ICV para {grupo} ({genero}) — revisar parser aseg.stats")
            continue  # Saltar esta combinación si no hay ICV
        
        # Seleccionar columnas que NO son subject, NO son ICV, y NO contienen "thickness"
        # Las columnas de grosor (thickness) NO se normalizan
        cols_to_normalize = [
            col for col in df.columns 
            if col not in ["subject", "ICV"]  # Excluir subject e ICV
            and "thickness" not in col.lower()  # Excluir grosores corticales
        ]
        
        print("\nColumnas a normalizar:", len(cols_to_normalize))  # Mostrar cantidad de columnas a normalizar
        
        # Normalizar cada columna de volumen dividiendo por el ICV del sujeto
        for col in cols_to_normalize:
            df[col] = df[col] / df["ICV"]  # Sobrescribe directamente la columna original
        
        print("Normalización por ICV completada (valores originales reemplazados)")
        
        # Convertir grupo a minúsculas para el nombre del archivo
        grupo_minuscula = grupo.lower()
        # Crear nombre del archivo CSV: volumenes_y_grosores_corticales_{grupo}_{genero}.csv
        output_csv = f"volumenes_y_grosores_corticales_{grupo_minuscula}_{genero}.csv"
        
        # Guardar el DataFrame como archivo CSV (sin incluir el índice)
        df.to_csv(output_csv, index=False)

# Mensaje final de finalización del proceso
print("\n" + "="*60)
print("PROCESO COMPLETAMENTE FINALIZADO")
print("="*60)


PROCESANDO GRUPO: AD - GÉNERO: M
Procesando: Sujeto 1... Hecho.
Procesando: Sujeto 2... Hecho.
Procesando: Sujeto 3... Hecho.
Procesando: Sujeto 4... Hecho.
Procesando: Sujeto 5... Hecho.
Procesando: Sujeto 6... Hecho.
Procesando: Sujeto 7... Hecho.
Procesando: Sujeto 8... Hecho.
Procesando: Sujeto 9... Hecho.
Procesando: Sujeto 10... Hecho.
Procesando: Sujeto 11... Hecho.
Procesando: Sujeto 12... Hecho.
Procesando: Sujeto 13... Hecho.
Procesando: Sujeto 14... Hecho.
Procesando: Sujeto 15... Hecho.
Procesando: Sujeto 16... Hecho.
Procesando: Sujeto 17... Hecho.
Procesando: Sujeto 18... Hecho.
Procesando: Sujeto 19... Hecho.
Procesando: Sujeto 20... Hecho.
Procesando: Sujeto 21... Hecho.
Procesando: Sujeto 22... Hecho.
Procesando: Sujeto 23... Hecho.
Procesando: Sujeto 24... Hecho.
Procesando: Sujeto 25... Hecho.
Procesando: Sujeto 26... Hecho.
Procesando: Sujeto 27... Hecho.
Procesando: Sujeto 28... Hecho.
Procesando: Sujeto 29... Hecho.
Procesando: Sujeto 30... Hecho.
Procesando: Suj

# 5) volúmenes_grosores_corticales_y_surfárea

In [5]:
import os
import pandas as pd
import glob

# Ruta base que contiene las carpetas de cada grupo
ruta_base_grupos = r"\\wsl.localhost\Ubuntu\home\joselito"

# Definir los grupos a procesar (Alzheimer, EMCI, LMCI, Controles Normales)
grupos = ["AD", "EMCI", "LMCI", "CN"]

# Definir los géneros a procesar (masculino y femenino)
generos = ["m", "f"]


# Función para rastrear y extraer volúmenes subcorticales e ICV desde aseg.stats
def parse_aseg(path):
    """
    Parsea el archivo aseg.stats de FreeSurfer para extraer volúmenes e ICV.
    
    Parámetros:
    -----------
    path : str
        Ruta al archivo aseg.stats
    
    Retorna:
    --------
    dict
        Diccionario con los volúmenes de las estructuras y el ICV
    """
    data = {}  # Diccionario para almacenar los volúmenes extraídos

    with open(path) as f:  # Abrir el archivo aseg.stats
        for line in f:  # Iterar línea por línea

            # Buscar la línea que contiene "EstimatedTotalIntraCranialVol"
            if line.startswith("#") and "EstimatedTotalIntraCranialVol" in line:
                parts = line.split(",")  # Dividir por comas (formato del archivo)
                try:
                    data["ICV"] = float(parts[3])  # El ICV está en la 4ª posición después de split por comas
                except:
                    pass  # Si falla, continuar sin guardar
                continue  # Saltar al siguiente ciclo

            if line.startswith("#"):  # Saltar líneas de comentario
                continue

            parts = line.split()  # Dividir la línea por espacios

            if len(parts) < 5:  # Las líneas válidas deben tener al menos 5 elementos
                continue

            try:
                name = parts[4]  # El nombre de la estructura está en la 5ª columna
                vol = float(parts[3])  # El volumen está en la 4ª columna
                data[name] = vol  # Guardar el volumen con el nombre de la estructura
            except:
                continue  # Si hay error, continuar con la siguiente línea

    return data  # Retornar el diccionario con todos los volúmenes

# Función para rastrear y extraer el área superficial y los grosores corticales desde aparc.stats
def parse_aparc(path, hemi):
    """
    Parsea el archivo aparc.stats de FreeSurfer para extraer área superficial
    y grosor cortical (NO extrae GrayVol).
    
    Parámetros:
    -----------
    path : str
        Ruta al archivo lh.aparc.stats o rh.aparc.stats
    hemi : str
        Hemisferio ("L" o "R")
    
    Retorna:
    --------
    dict
        Diccionario con área superficial y grosor por región
    """
    data = {}  # Diccionario para almacenar los grosores y áreas superficiales

    if not os.path.exists(path):  # Verificar si el archivo existe
        return data  # Si no existe, retornar diccionario vacío

    with open(path) as f:  # Abrir el archivo aparc.stats
        for line in f:  # Iterar línea por línea

            if line.startswith("#"):  # Saltar líneas de comentario
                continue

            parts = line.split()  # Dividir la línea por espacios

            # Necesitamos hasta ThickAvg (columna 4)
            if len(parts) < 5:  # Las líneas válidas deben tener al menos 5 elementos
                continue

            try:
                region = parts[0]  # El nombre de la región está en la 1ª columna

                surf_area = float(parts[2])  # Área superficial en la 3ª columna
                thick_avg = float(parts[4])  # Grosor cortical promedio en la 5ª columna

                # Guardar el área superficial con el formato: región_hemisferio_SurfArea
                data[f"{region}_{hemi}_SurfArea"] = surf_area
                # Guardar el grosor con el formato: región_hemisferio_thickness
                data[f"{region}_{hemi}_thickness"] = thick_avg

            except:
                continue  # Si hay error, continuar con la siguiente línea

    return data  # Retornar el diccionario con todos los grosores y áreas

# Procesar cada combinación de grupo y género por separado
for grupo in grupos:  # Iterar sobre cada grupo (AD, EMCI, LMCI, CN)
    for genero in generos:  # Iterar sobre cada género (m, f)
        print(f"\n{'='*60}")
        print(f"PROCESANDO GRUPO: {grupo} - GÉNERO: {genero.upper()}")
        print(f"{'='*60}")
        
        # Construir la ruta a la carpeta de FreeSurfer para este grupo y género
        # Estructura: FMRIPREP_{GRUPO}_{GENERO}_MRI/sourcedata/freesurfer
        freesurfer_dir = f"FMRIPREP_{grupo}_{genero}_MRI"
        fs_root = os.path.join(ruta_base_grupos, freesurfer_dir, "sourcedata", "freesurfer")
        
        # Verificar si la ruta existe
        if not os.path.exists(fs_root):
            print(f"ADVERTENCIA: Ruta no encontrada para {grupo} ({genero}): {fs_root}")
            continue  # Saltar esta combinación si no existe la ruta
        
        # Obtener lista de sujetos (carpetas que empiezan con "sub-")
        subjects = glob.glob(os.path.join(fs_root, "sub-*"))
        
        if not subjects:  # Si no hay sujetos, mostrar mensaje y continuar
            print(f"No se encontraron sujetos para {grupo} ({genero})")
            continue
        
        rows = []  # Lista para almacenar los datos de todos los sujetos
        
        for idx_sujeto, sub in enumerate(subjects, start=1):  # Iterar sobre cada sujeto
            sub_id = os.path.basename(sub)  # Extraer el ID del sujeto (nombre de la carpeta)
            print(f"Procesando: Sujeto {idx_sujeto}...", end=" ", flush=True)
            stats_dir = os.path.join(sub, "stats")  # Directorio stats de FreeSurfer
            
            # Construir rutas a los archivos de estadísticas
            aseg_path = os.path.join(stats_dir, "aseg.stats")  # Volúmenes subcorticales
            lh_path = os.path.join(stats_dir, "lh.aparc.stats")  # Hemisferio izquierdo
            rh_path = os.path.join(stats_dir, "rh.aparc.stats")  # Hemisferio derecho
            
            # Verificar si existe el archivo aseg.stats (obligatorio)
            if not os.path.exists(aseg_path):
                print(f"Saltando {sub_id} — sin aseg.stats")
                continue  # Saltar este sujeto si no tiene aseg.stats
            
            # Inicializar fila con el ID del sujeto
            row = {"subject": sub_id}
            
            aseg_data = parse_aseg(aseg_path)  # Extraer volúmenes e ICV
            icv = aseg_data.get("ICV", None)  # Obtener el ICV si existe
            
            if icv is None:
                # Si no hay ICV, advertir y guardar volúmenes sin normalizar
                print(f"Advertencia: {sub_id} sin ICV, se omite normalización")
                row.update(aseg_data)  # Añadir todos los volúmenes sin normalizar
            else:
                # Normalizar todas las columnas de aseg excepto ICV
                for k, v in aseg_data.items():
                    if k == "ICV":
                        row[k] = v  # ICV se guarda sin normalizar
                    else:
                        row[k] = v / icv  # Volúmenes normalizados por ICV
            
            aparc_lh = parse_aparc(lh_path, "L")  # Extraer datos del hemisferio izquierdo
            aparc_rh = parse_aparc(rh_path, "R")  # Extraer datos del hemisferio derecho
            
            # No se extrae GrayVol, solo SurfArea y ThickAvg (NO se normalizan)
            # Unir los diccionarios de ambos hemisferios y añadir a la fila
            for k, v in {**aparc_lh, **aparc_rh}.items():
                row[k] = v  # Áreas y grosores se guardan sin normalizar
            
            rows.append(row)  # Añadir la fila a la lista de resultados
            print("Hecho.")
        
        if not rows:  # Si no se procesó ningún sujeto, continuar
            print(f"No se procesaron sujetos para {grupo} ({genero})")
            continue
        
        df = pd.DataFrame(rows)  # Convertir lista de diccionarios a DataFrame de pandas
        
        # Convertir grupo a minúsculas para el nombre del archivo
        grupo_minuscula = grupo.lower()
        # Crear nombre del archivo CSV: volumenes_grosores_corticales_y_surfarea_{grupo}_{genero}.csv
        output_csv = f"volumenes_grosores_corticales_y_surfarea_{grupo_minuscula}_{genero}.csv"
        
        # Guardar el DataFrame como archivo CSV (sin incluir el índice)
        df.to_csv(output_csv, index=False)

# Mensaje final de finalización del proceso
print("\n" + "="*60)
print("PROCESO COMPLETAMENTE FINALIZADO")
print("="*60)


PROCESANDO GRUPO: AD - GÉNERO: M
Procesando: Sujeto 1... Hecho.
Procesando: Sujeto 2... Hecho.
Procesando: Sujeto 3... Hecho.
Procesando: Sujeto 4... Hecho.
Procesando: Sujeto 5... Hecho.
Procesando: Sujeto 6... Hecho.
Procesando: Sujeto 7... Hecho.
Procesando: Sujeto 8... Hecho.
Procesando: Sujeto 9... Hecho.
Procesando: Sujeto 10... Hecho.
Procesando: Sujeto 11... Hecho.
Procesando: Sujeto 12... Hecho.
Procesando: Sujeto 13... Hecho.
Procesando: Sujeto 14... Hecho.
Procesando: Sujeto 15... Hecho.
Procesando: Sujeto 16... Hecho.
Procesando: Sujeto 17... Hecho.
Procesando: Sujeto 18... Hecho.
Procesando: Sujeto 19... Hecho.
Procesando: Sujeto 20... Hecho.
Procesando: Sujeto 21... Hecho.
Procesando: Sujeto 22... Hecho.
Procesando: Sujeto 23... Hecho.
Procesando: Sujeto 24... Hecho.
Procesando: Sujeto 25... Hecho.
Procesando: Sujeto 26... Hecho.
Procesando: Sujeto 27... Hecho.
Procesando: Sujeto 28... Hecho.
Procesando: Sujeto 29... Hecho.
Procesando: Sujeto 30... Hecho.
Procesando: Suj